In [1]:
import pandas as pd
import os
import torch
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from torch.utils.data import Dataset
from PIL import Image
import numpy as np
from torchvision import models
import torch.nn as nn
from torch.utils.data import ConcatDataset
import torch.optim as optim
import random

In [2]:
## Good implementation of the upload

import kagglehub
import shutil
import os

# 1. Download via kagglehub (goes to default cache)
cache_path = kagglehub.dataset_download("puneet6060/intel-image-classification")
print("Temporary path :", cache_path)

# 2. The folder where you really want your data
dossier_final = "./my_intel_dataset"

# 3. We copy the data to the final folder
if os.path.exists(dossier_final):
    # If the folder already exists, we delete it to avoid conflicts
    shutil.rmtree(dossier_final)

# Use shutil.copytree instead of shutil.move because cache_path is read-only
shutil.copytree(cache_path, dossier_final)

print("Success ! The dataset is now directly in :", dossier_final)

Using Colab cache for faster access to the 'intel-image-classification' dataset.
Temporary path : /kaggle/input/intel-image-classification
Success ! The dataset is now directly in : ./my_intel_dataset


dataset : https://www.kaggle.com/datasets/puneet6060/intel-image-classification?select=seg_train

Create the label files

In [3]:
### AJOUTE PAR KEVI POUR ADAPTER LE CODE A COLAB

from pathlib import Path
import shutil # Import shutil for moving files
import os
import pandas as pd

#dataset_train_path = '/Users/laurageneaulibourne/Downloads/S4/ia/exam/dataset/seg_train/seg_train'
dataset_train_path="/content/my_intel_dataset/seg_train/seg_train"

#dataset__path = '/Users/laurageneaulibourne/Downloads/S4/ia/exam/dataset/seg_test/seg_test'
dataset_test_path='/content/my_intel_dataset/seg_test/seg_test'

classe = {'buildings' : 0, 'forest' : 1, 'glacier' : 2, 'mountain' : 3, 'sea' : 4, 'street' : 5}

for c in classe.keys():
    label_train = []
    label_test = []

    class_train_path = os.path.join(dataset_train_path, c)
    class_test_path = os.path.join(dataset_test_path, c)

    img_train_path = class_train_path +'/img'
    img_test_path = class_test_path+'/img'
    print(img_train_path)
    print(img_test_path)

    dossier_train = Path(img_train_path)
    dossier_train.mkdir(parents=True, exist_ok=True)

    dossier_test = Path(img_test_path)
    dossier_test.mkdir(parents=True, exist_ok=True)


    # Move images from the class directory to the 'img' subdirectory
    for f in os.listdir(class_train_path):
        if f.lower().endswith(('.png', '.jpg', '.jpeg', '.gif', '.bmp')):
            shutil.move(os.path.join(class_train_path, f), img_train_path)

    for f in os.listdir(class_test_path):
        if f.lower().endswith(('.png', '.jpg', '.jpeg', '.gif', '.bmp')):
            shutil.move(os.path.join(class_test_path, f), img_test_path)

    # Now, list files from the 'img' subdirectories, which now contain the images
    for imag in os.listdir(img_train_path):
        if not imag.startswith('.'): # avoid .DS_Store
            label_train.append({'image_name' : imag, 'label' : classe[c]})


    for imag in os.listdir(img_test_path):
        if not imag.startswith('.'):
            label_test.append({'image_name' : imag, 'label' : classe[c]})


    df_train = pd.DataFrame(label_train)
    df_train.to_csv(os.path.join(class_train_path, f"{classe[c]}_label_train.csv"), index=False)
    df_test = pd.DataFrame(label_test)
    df_test.to_csv(os.path.join(class_test_path, f"{classe[c]}_label_test.csv"), index=False)

/content/my_intel_dataset/seg_train/seg_train/buildings/img
/content/my_intel_dataset/seg_test/seg_test/buildings/img
/content/my_intel_dataset/seg_train/seg_train/forest/img
/content/my_intel_dataset/seg_test/seg_test/forest/img
/content/my_intel_dataset/seg_train/seg_train/glacier/img
/content/my_intel_dataset/seg_test/seg_test/glacier/img
/content/my_intel_dataset/seg_train/seg_train/mountain/img
/content/my_intel_dataset/seg_test/seg_test/mountain/img
/content/my_intel_dataset/seg_train/seg_train/sea/img
/content/my_intel_dataset/seg_test/seg_test/sea/img
/content/my_intel_dataset/seg_train/seg_train/street/img
/content/my_intel_dataset/seg_test/seg_test/street/img


create our dataset

In [4]:
#For each task we will do a classification on 2 classes, so we will create a new dataset with only 2 classes by merging our 2 directories and same thing for the file with the labels.

def task(A,B, idx):
    '''A and B are the 2 classes of the idx task'''
    path_img_A_train = os.path.join(dataset_train_path, A, 'img')
    path_img_B_train = os.path.join(dataset_train_path, B, 'img')
    path_img_AB_train = os.path.join(dataset_train_path, f'df_{A}_{B}', 'img')
    path_img_A_test = os.path.join(dataset_test_path, A, 'img')
    path_img_B_test = os.path.join(dataset_test_path, B, 'img')
    path_img_AB_test = os.path.join(dataset_test_path, f'df_{A}_{B}', 'img')

    #create the directory if it does not exist
    os.makedirs(path_img_AB_train, exist_ok=True)
    os.makedirs(path_img_AB_test, exist_ok=True)

    def move(init, final):
        for file in os.listdir(init):
            if not file.startswith('.'):
                source_file = os.path.join(init, file)
                dest_file = os.path.join(final, file)
         #       os.rename(source_file, dest_file)
                # Shutil fait la meme chose que remove mais en mieux
                shutil.move(source_file,dest_file)



    move(path_img_A_train, path_img_AB_train)
    move(path_img_B_train, path_img_AB_train)

    move(path_img_A_test, path_img_AB_test)
    move(path_img_B_test, path_img_AB_test)

    label_A_train = pd.read_csv(os.path.join(dataset_train_path, A, f"{classe[A]}_label_train.csv"))
    label_B_train = pd.read_csv(os.path.join(dataset_train_path, B, f"{classe[B]}_label_train.csv"))
    label_A_train['label'] = 0
    label_B_train['label'] = 1
    label_AB_train = pd.concat([label_A_train, label_B_train], ignore_index=True)
    label_AB_train['task_id'] = idx
    label_AB_train.to_csv(os.path.join(dataset_train_path,f'df_{A}_{B}', 'label_train.csv'), index=False)



    label_A_test = pd.read_csv(os.path.join(dataset_test_path, A, f"{classe[A]}_label_test.csv"))
    label_B_test = pd.read_csv(os.path.join(dataset_test_path, B, f"{classe[B]}_label_test.csv"))
    label_A_test['label'] = 0
    label_B_test['label'] = 1
    label_AB_test = pd.concat([label_A_test, label_B_test], ignore_index=True)
    label_AB_test['task_id'] = idx
    label_AB_test.to_csv(os.path.join(dataset_test_path, f'df_{A}_{B}', 'label_test.csv'), index=False)

    return (path_img_AB_train, path_img_AB_test, label_AB_train, label_AB_test)

path_img_AB_train, path_img_AB_test, label_AB_train, label_AB_test = task('buildings', 'sea', 0)
path_img_CD_train, path_img_CD_test, label_CD_train, label_CD_test = task('forest', 'mountain', 1)

label_ABCD_train=pd.concat([label_AB_train,label_CD_train], ignore_index=True)
label_ABCD_train.to_csv(os.path.join(dataset_train_path, 'label_train_ABCD.csv'), index=False)

label_ABCD_test=pd.concat([label_AB_test, label_CD_test], ignore_index=True)
label_ABCD_test.to_csv(os.path.join(dataset_test_path, 'label_test_ABCD.csv'), index=False)




In [5]:
print(path_img_AB_train)
print(path_img_AB_test)

/content/my_intel_dataset/seg_train/seg_train/df_buildings_sea/img
/content/my_intel_dataset/seg_test/seg_test/df_buildings_sea/img


Repartition of the dataset through the 6 classes

In [6]:

def repartition(label_train, label_test, classe_0, classe_1):

    train_tot = len(label_train) # total number of images in the train for this task
    test_tot  = len(label_test) # total number of images in the test for this task

    nb_0_train = (label_train['label'] == 0).sum() #number of images in the train for the class 0
    nb_1_train = (label_train['label'] == 1).sum() #number of images in the train for the class 1
    nb_0_test  = (label_test['label'] == 0).sum() #number of images in the test for the class 0
    nb_1_test  = (label_test['label'] == 1).sum() #number of images in the test for the class 1

    print(f"Task : {label_train['task_id'].iloc[0]} \n")
    print(f" Total images in the train for the task {classe_0}/{classe_1} : {train_tot}")
    print(f" Pourcentage of images in the train for the class {classe_0} : {nb_0_train * 100/ train_tot}%")
    print(f" Pourcentage of images in the train for the class {classe_1} : {nb_1_train * 100/ train_tot}%")

    print(f" Total images in the test for the task {classe_0}/{classe_1} : {test_tot}")
    print(f" Pourcentage of images in the test for the class {classe_0} : {nb_0_test * 100/ test_tot}%")
    print(f" Pourcentage of images in the test for the class {classe_1} : {nb_1_test * 100/ test_tot}% \n")


repartition(label_AB_train, label_AB_test, 'buildings', 'sea')
repartition(label_CD_train, label_CD_test, 'forest', 'mountain')



Task : 0 

 Total images in the train for the task buildings/sea : 4465
 Pourcentage of images in the train for the class buildings : 49.07054871220605%
 Pourcentage of images in the train for the class sea : 50.92945128779395%
 Total images in the test for the task buildings/sea : 947
 Pourcentage of images in the test for the class buildings : 46.14572333685322%
 Pourcentage of images in the test for the class sea : 53.85427666314678% 

Task : 1 

 Total images in the train for the task forest/mountain : 4783
 Pourcentage of images in the train for the class forest : 47.48066067321765%
 Pourcentage of images in the train for the class mountain : 52.51933932678235%
 Total images in the test for the task forest/mountain : 999
 Pourcentage of images in the test for the class forest : 47.447447447447445%
 Pourcentage of images in the test for the class mountain : 52.552552552552555% 



In [ ]:
"""

def portion_dataset(images,label, portion): #returns a portion of the dataset (images and labels) given a portion value between 0 and 1.

    nb = int(len(label) * portion) #number of images in the dataset that we will keep in order to avoid catastrophic forgetting during the new task.

    idx = np.random.choice(len(label), size=nb, replace=False) #indexes of the images that we will keep.

    img_portion = images[idx]
    label_portion = label[idx]

    return img_portion, label_portion

"""

In [7]:
class CustomImageDataset(Dataset):

    def __init__(self, annotations_file, img_dir, portion, img_transform=None): # portion is the pourcentage of the dataset that we will keep in order to avoid catastrophic forgetting during the new task.

        self.img_dir = img_dir
        self.img_transform = img_transform

        if portion < 1.0:
            self.img_labels = annotations_file.sample(frac=portion, random_state=42).reset_index(drop=True)
        else:
            self.img_labels = annotations_file

    def __len__(self): #Return the number of samples in the dataset.
        return len(self.img_labels)

    def __getitem__(self, idx): #For a given index, it returns the sample (image, label) associated to it.

        # Get the file path of the image by combining the directory and the image filename from the labels DataFrame
        img_path = os.path.join(self.img_dir, self.img_labels.iloc[idx, 0])

        # Read the image from the specified file path and convert it to a tensor
        image = Image.open(img_path)

        # Retrieve the corresponding label for the image from the labels DataFrame
        label = self.img_labels.iloc[idx, 1]

        task = self.img_labels.iloc[idx, 2]

        # If a transform function is provided, apply it to the image
        if self.img_transform:
            image = self.img_transform(image)
            # If a target_transform function is provided, apply it to the label

        # Return the transformed image and label as a tuple
        return image, label, task


In [8]:
# List of transformations that will be applied to images
transform = transforms.Compose([
    transforms.Resize((150, 150)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],std=[0.229, 0.224, 0.225])
])


In [9]:

dataset_train_AB_full = CustomImageDataset(annotations_file= label_AB_train, img_dir = path_img_AB_train, portion = 1 , img_transform=transform)
dataset_test_AB_full = CustomImageDataset(annotations_file= label_AB_test, img_dir = path_img_AB_test, portion = 1 , img_transform=transform)

train_dataloader_AB_full = DataLoader(dataset_train_AB_full, batch_size=64, shuffle=True)
test_dataloader_AB_full= DataLoader(dataset_test_AB_full, batch_size=64, shuffle=True)


#portion = 0.5 #portion of the dataset of the previous dataset we want to keep for avoiding the catastrophic forgetting during the new task.
portion=1


dataset_train_AB_part = CustomImageDataset(annotations_file= label_AB_train, img_dir = path_img_AB_train, portion = portion , img_transform=transform)
dataset_test_AB_part = CustomImageDataset(annotations_file= label_AB_test, img_dir = path_img_AB_test, portion = portion, img_transform=transform)

train_dataloader_AB_part = DataLoader(dataset_train_AB_part, batch_size=64, shuffle=True)
test_dataloader_AB_part= DataLoader(dataset_test_AB_part, batch_size=64,  shuffle=True)





dataset_train_CD_full = CustomImageDataset(annotations_file= label_CD_train, img_dir = path_img_CD_train, portion = 1 , img_transform=transform)
dataset_test_CD_full = CustomImageDataset(annotations_file= label_CD_test, img_dir = path_img_CD_test, portion = 1, img_transform=transform)

train_dataloader_CD_full = DataLoader(dataset_train_CD_full, batch_size=64, shuffle=True)
test_dataloader_CD_full= DataLoader(dataset_test_CD_full, batch_size=64, shuffle=True)






dataset_train_ABCD = ConcatDataset([dataset_train_AB_part, dataset_train_CD_full])
dataset_test_ABCD = ConcatDataset([dataset_test_AB_part, dataset_test_CD_full])

train_dataloader_ABCD = DataLoader(dataset_train_ABCD, batch_size=64, shuffle=True)
test_dataloader_ABCD= DataLoader(dataset_test_ABCD, batch_size=64, shuffle=True)


In [17]:

"""
def reduce_dataset(dataset, portion):
    # portion = the portion of the dataset of the previous tasks we want to keep for avoiding the catastrophic forgetting during the new task.
    nb = int(len(dataset) * portion) #number of images to keep

    # 2. Générer des indices aléatoires uniques
    indices = np.random.choice(len(dataset), size=nb, replace=False)
    return Subset(dataset, indices)

# 3. Créer le sous-jeu de données et le passer au DataLoader standard
dataset_reduit = Subset(dataset_train, indices)
train_loader = DataLoader(dataset_reduit, batch_size=64, shuffle=True)

"""

'\ndef reduce_dataset(dataset, portion):\n    # portion = the portion of the dataset of the previous tasks we want to keep for avoiding the catastrophic forgetting during the new task.\n    nb = int(len(dataset) * portion) #number of images to keep\n\n    # 2. Générer des indices aléatoires uniques\n    indices = np.random.choice(len(dataset), size=nb, replace=False)\n    return Subset(dataset, indices)\n\n# 3. Créer le sous-jeu de données et le passer au DataLoader standard\ndataset_reduit = Subset(dataset_train, indices)\ntrain_loader = DataLoader(dataset_reduit, batch_size=64, shuffle=True)\n\n'

In [18]:
basic_model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
num_features = basic_model.fc.in_features
basic_model.fc = nn.Linear(num_features, 2)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 148MB/s]


In [10]:
def train_loop_basic(dataloader, model, loss, optimizer, num_epochs, device):
    size = len(dataloader.dataset)
    model = model.to(device)
    model.train()

    for epoch in range(num_epochs):

        epoch_loss = 0.0
        dic_acc = {"A_true": 0,"A_tot": 0,  "B_true": 0, "B_tot" : 0}  # accuracy dictionary for each class

        for image, label, _ in dataloader:

            #put image and label on the same device
            image= image.to(device)
            label= label.to(device)

            optimizer.zero_grad()  # Set gradient to zero

            # Compute prediction and loss
            pred = model(image)

            batch_loss = loss(pred, label)
            epoch_loss += batch_loss.item()

            pred_class = pred.argmax(dim=1)
            pred_classA = (pred_class == 0)
            label_A = (label == 0)
            dic_acc["A_true"] += (pred_classA & label_A).sum().item()
            dic_acc["A_tot"] += label_A.sum().item()

            pred_classB = (pred_class == 1)
            label_B = (label == 1)
            dic_acc["B_true"] += (pred_classB & label_B).sum().item()
            dic_acc["B_tot"] += label_B.sum().item()

            # Backpropagation
            batch_loss.backward()
            optimizer.step()

        epoch_loss = epoch_loss / size


        print(f'Epoch {epoch+1}/{num_epochs}, Total Mean Loss: {epoch_loss}, '
              f'Accuracy A: {(dic_acc["A_true"] * 100 / dic_acc["A_tot"] if dic_acc["A_tot"] > 0 else 0.0)} %, '
              f'B: {(dic_acc["B_true"] * 100 / dic_acc["B_tot"] if dic_acc["B_tot"] > 0 else 0.0)} %, '
    )




In [11]:
def test_loop_basic(dataloader, model, loss, device):
    size = len(dataloader.dataset)
    model = model.to(device)
    model.eval()

    with torch.no_grad():

        test_loss = 0.0
        dic_acc = {"A_true": 0,"A_tot": 0,  "B_true": 0, "B_tot" : 0}  # accuracy dictionary for each class

        for image, label, _ in dataloader:

            #put image and label on the same device
            image= image.to(device)
            label= label.to(device)

            # Compute prediction and loss
            pred = model(image)

            batch_loss = loss(pred, label)
            test_loss += batch_loss.item()* label.size(0)

            pred_class = pred.argmax(dim=1)
            pred_classA = (pred_class == 0)
            label_A = (label == 0)
            dic_acc["A_true"] += (pred_classA & label_A).sum().item()
            dic_acc["A_tot"] += label_A.sum().item()

            pred_classB = (pred_class == 1)
            label_B = (label == 1)
            dic_acc["B_true"] += (pred_classB & label_B).sum().item()
            dic_acc["B_tot"] += label_B.sum().item()

            # Backpropagation
            batch_loss.backward()
            optimizer.step()

        test_loss = test_loss / size


        print(f'Test Loss: {test_loss}, '
              f'Accuracy A: {(dic_acc["A_true"] * 100 / dic_acc["A_tot"] if dic_acc["A_tot"] > 0 else 0.0)} %, '
              f'B: {(dic_acc["B_true"] * 100 / dic_acc["B_tot"] if dic_acc["B_tot"] > 0 else 0.0)} %, '
    )




In [12]:
class rehearsal(nn.Module):
# Constructor method to initialize layers of the network def __init__(self):
        def __init__(self,num_classe=2):
                super().__init__()
                self.num_classe = num_classe
                self.model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT) # we consider a pretrained model
                self.num_features = self.model.fc.in_features
                self.model.fc = nn.Identity()
                self.heads = nn.ModuleDict()

        def add_task(self, num_task):
                num_task = str(num_task)
                if num_task not in self.heads:
                        self.heads[num_task] = nn.Linear(self.num_features, self.num_classe)

        # Forward pass of the network
        def forward(self, x, num_task):
                features = self.model(x)  # features pour toutes les images du batch
                outputs = torch.zeros(x.size(0), self.num_classe, device=x.device)

                for t in torch.unique(num_task):
                        t_str = str(t.item())
                        if t_str not in self.heads:
                                raise ValueError(f"Task {t_str} not found in the model.")
                        mask = (num_task == t)
                        outputs[mask] = self.heads[t_str](features[mask])

                return outputs

'''

        def forward(self, x, num_task):
                x = self.model(x)
                num_task = num_task[0].item() if isinstance(num_task, torch.Tensor) else num_task
                num_task = str(num_task)
                if num_task in self.heads:
                        return self.heads[num_task](x)
                else:
                        raise ValueError(f"Task {num_task} not found in the model.")
'''

'\n\n        def forward(self, x, num_task):\n                x = self.model(x)\n                num_task = num_task[0].item() if isinstance(num_task, torch.Tensor) else num_task\n                num_task = str(num_task)\n                if num_task in self.heads:\n                        return self.heads[num_task](x)\n                else:\n                        raise ValueError(f"Task {num_task} not found in the model.")\n'

In [13]:
def train_loop_rehearsal(dataloader, model, loss, optimizer, num_epochs, device):

    model = model.to(device)
    model.train()

    for epoch in range(num_epochs):

        epoch_lossAB = 0.0
        epoch_lossCD = 0.0
        epoch_loss_tot = 0.0
        dic_acc = {"A_true": 0,"A_tot": 0,  "B_true": 0, "B_tot" : 0,  "C_true": 0, "C_tot" : 0, "D_true": 0, "D_tot": 0}  # accuracy dictionary for each class


        for image, label, task in dataloader:


            batch_lossAB = 0.0
            batch_lossCD = 0.0
            batch_loss_tot = 0.0
            #put image and label on the same device
            image= image.to(device)
            label= label.to(device)

            optimizer.zero_grad()  # Set gradient to zero

            # Compute prediction and loss
            pred = model(image, task)

            mask_AB = (task == 0)
            mask_CD = (task == 1)

            if mask_AB.sum() > 0: # at least one image from the 1st task in the batch
                pred_AB = pred[mask_AB]
                label_AB = label[mask_AB]

                batch_lossAB = loss(pred_AB, label_AB)
                epoch_lossAB += batch_lossAB.item()* label_AB.size(0)

                # For each image, choose the class with the higher score
                pred_classAB = pred_AB.argmax(dim=1)
                pred_classA = (pred_classAB == 0)
                label_A = (label_AB == 0)
                dic_acc["A_true"] += (pred_classA & label_A).sum().item()
                dic_acc["A_tot"] += label_A.sum().item()

                pred_classB = (pred_classAB == 1)
                label_B = (label_AB == 1)
                dic_acc["B_true"] += (pred_classB & label_B).sum().item()
                dic_acc["B_tot"] += label_B.sum().item()


            if mask_CD.sum() > 0: # at least one image from the 2nd task in the batch
                pred_CD = pred[mask_CD]
                label_CD = label[mask_CD]

                batch_lossCD = loss(pred_CD, label_CD)
                epoch_lossCD += batch_lossCD.item()* label_CD.size(0)

                pred_classCD = pred_CD.argmax(dim=1)
                pred_classC = (pred_classCD == 0)
                label_C = (label_CD == 0)
                dic_acc["C_true"] += (pred_classC & label_C).sum().item()
                dic_acc["C_tot"] += label_C.sum().item()

                pred_classD = (pred_classCD == 1)
                label_D = (label_CD == 1)
                dic_acc["D_true"] += (pred_classD & label_D).sum().item()
                dic_acc["D_tot"] += label_D.sum().item()


            batch_loss_tot = batch_lossAB + batch_lossCD
            epoch_loss_tot += batch_loss_tot.item()

            # Backpropagation
            batch_loss_tot.backward()
            optimizer.step()

        nb_AB = dic_acc["A_tot"] + dic_acc["B_tot"]
        nb_CD = dic_acc["C_tot"] + dic_acc["D_tot"]

        epoch_lossAB = epoch_lossAB / nb_AB
        if nb_CD > 0:
            epoch_lossCD = epoch_lossCD / nb_CD
        else :
            epoch_lossCD = 0.0
        epoch_loss = epoch_lossAB + epoch_lossCD


        print(f'Epoch {epoch+1}/{num_epochs}, Total Mean Loss: {epoch_loss}, 1st task Loss: {epoch_lossAB}, 2nd task Loss: {epoch_lossCD}, '
              f'Accuracy A: {(dic_acc["A_true"] * 100 / dic_acc["A_tot"] if dic_acc["A_tot"] > 0 else 0.0)} %, '
              f'B: {(dic_acc["B_true"] * 100 / dic_acc["B_tot"] if dic_acc["B_tot"] > 0 else 0.0)} %, '
              f'C: {(dic_acc["C_true"] * 100 / dic_acc["C_tot"] if dic_acc["C_tot"] > 0 else 0.0)} %, '
              f'D: {(dic_acc["D_true"] * 100 / dic_acc["D_tot"] if dic_acc["D_tot"] > 0 else 0.0)} %')




In [14]:
def test_loop_rehearsal(dataloader, model, loss, device):

    model = model.to(device)
    model.eval()
    loss_test = 0.0
    loss_testAB = 0.0
    loss_testCD = 0.0

    dic_acc = {"A_true": 0,"A_tot": 0,  "B_true": 0, "B_tot" : 0,  "C_true": 0, "C_tot" : 0, "D_true": 0, "D_tot": 0}  # accuracy dictionary for each class

    with torch.no_grad():


        for image, label, task in dataloader:

            batch_lossAB = 0.0
            batch_lossCD = 0.0
            batch_loss_tot = 0.0

            #put image and label on the same device
            image= image.to(device)
            label= label.to(device)

            # Compute prediction and loss
            pred = model(image, task)

            mask_AB = (task == 0)
            mask_CD = (task == 1)

            if mask_AB.sum() > 0: # at least one image from the 1st task in the batch
                pred_AB = pred[mask_AB]
                label_AB = label[mask_AB]

                batch_lossAB = loss(pred_AB, label_AB)
                loss_testAB += batch_lossAB.item()* label_AB.size(0)

                # For each image, choose the class with the higher score
                pred_classAB = pred_AB.argmax(dim=1)
                pred_classA = (pred_classAB == 0)
                label_A = (label_AB == 0)
                dic_acc["A_true"] += (pred_classA & label_A).sum().item()
                dic_acc["A_tot"] += label_A.sum().item()

                pred_classB = (pred_classAB == 1)
                label_B = (label_AB == 1)
                dic_acc["B_true"] += (pred_classB & label_B).sum().item()
                dic_acc["B_tot"] += label_B.sum().item()



            if mask_CD.sum() > 0: # at least one image from the 2nd task in the batch
                pred_CD = pred[mask_CD]
                label_CD = label[mask_CD]

                batch_lossCD = loss(pred_CD, label_CD)
                loss_testCD += batch_lossCD.item()* label_CD.size(0)

                pred_classCD = pred_CD.argmax(dim=1)
                pred_classC = (pred_classCD == 0)
                label_C = (label_CD == 0)
                dic_acc["C_true"] += (pred_classC & label_C).sum().item()
                dic_acc["C_tot"] += label_C.sum().item()

                pred_classD = (pred_classCD == 1)
                label_D = (label_CD == 1)
                dic_acc["D_true"] += (pred_classD & label_D).sum().item()
                dic_acc["D_tot"] += label_D.sum().item()

        nb_AB = dic_acc["A_tot"] + dic_acc["B_tot"]
        nb_CD = dic_acc["C_tot"] + dic_acc["D_tot"]

        loss_testAB = loss_testAB / nb_AB
        if nb_CD > 0:
            loss_testCD = loss_testCD / nb_CD
        else:
            loss_testCD = 0.0

        loss_test =  loss_testAB + loss_testCD


        print(f'Total Mean Loss per images: {loss_test}, 1st task Loss: {loss_testAB}, 2nd task Loss: {loss_testCD}',
              f'Accuracy A: {(dic_acc["A_true"] * 100 / dic_acc["A_tot"] if dic_acc["A_tot"] > 0 else 0.0)} %, '
              f'B: {(dic_acc["B_true"] * 100 / dic_acc["B_tot"] if dic_acc["B_tot"] > 0 else 0.0)} %, '
              f'C: {(dic_acc["C_true"] * 100 / dic_acc["C_tot"] if dic_acc["C_tot"] > 0 else 0.0)} %, '
              f'D: {(dic_acc["D_true"] * 100 / dic_acc["D_tot"] if dic_acc["D_tot"] > 0 else 0.0)} %')




In [15]:
model_rehearsal = rehearsal()
model_rehearsal.add_task(0)
model_rehearsal.add_task(1)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 130MB/s]


In [16]:
torch.backends.mps.is_available()

False

In [17]:
if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')

In [18]:
loss = nn.CrossEntropyLoss()
optimizer = optim.SGD(model_rehearsal.parameters(), lr=0.001, momentum=0.9)
num_epochs = 1

In [28]:
train_loop_rehearsal(train_dataloader_AB_full , model_rehearsal, loss, optimizer, num_epochs, device)

KeyboardInterrupt: 

In [ ]:
test_loop_rehearsal(test_dataloader_AB_full, model_rehearsal, loss, device)

Total Mean Loss per images: 0.016067508288483117, 1st task Loss: 0.016067508288483117, 2nd task Loss: 0.0 Accuracy A: 99.54233409610984 %, B: 99.41176470588235 %, C: 0.0 %, D: 0.0 %


In [ ]:
train_loop_rehearsal(train_dataloader_ABCD , model_rehearsal, loss, optimizer, num_epochs, device)

Epoch 1/10, Total Mean Loss: 0.08901499142016442, 1st task Loss: 0.005147421530669673, 2nd task Loss: 0.08386756988949474, Accuracy A: 100.0 %, B: 99.73637961335676 %, C: 96.60942316160282 %, D: 97.53184713375796 %
Epoch 2/10, Total Mean Loss: 0.021413784622962524, 1st task Loss: 0.003376521601888509, 2nd task Loss: 0.018037263021074015, Accuracy A: 100.0 %, B: 99.82425307557118 %, C: 99.1193306913254 %, D: 99.8407643312102 %
Epoch 3/10, Total Mean Loss: 0.011933158379217776, 1st task Loss: 0.0009449114101134845, 2nd task Loss: 0.010988246969104292, Accuracy A: 100.0 %, B: 100.0 %, C: 99.47159841479524 %, D: 99.96019108280255 %
Epoch 4/10, Total Mean Loss: 0.007936676764109963, 1st task Loss: 0.0011202913856811969, 2nd task Loss: 0.006816385378428765, Accuracy A: 100.0 %, B: 100.0 %, C: 99.77983267283135 %, D: 100.0 %
Epoch 5/10, Total Mean Loss: 0.006206547326250809, 1st task Loss: 0.0007172986811530145, 2nd task Loss: 0.005489248645097795, Accuracy A: 100.0 %, B: 100.0 %, C: 99.77983

In [ ]:
test_loop_rehearsal(test_dataloader_ABCD, model_rehearsal, loss, device)

Total Mean Loss per images: 0.022609890823493378, 1st task Loss: 0.011621779697753302, 2nd task Loss: 0.010988111125740075 Accuracy A: 99.57627118644068 %, B: 99.15966386554622 %, C: 99.78902953586498 %, D: 99.42857142857143 %


A-GEM with better samples

In [19]:
def project_gradient(g, g_ref):

    dot_product = torch.dot(g, g_ref)
    if dot_product >= 0:
        return g
    else:
    #g_update = g - ( (g^t * g_ref) / (g_ref^T*g_ref) ) * g_ref but here we don't apply the transpose because g is already a vector
        dot_product_g_ref = torch.dot(g_ref, g_ref) + 1e-10 # to avoid division by zero
        g_update = g - (dot_product / dot_product_g_ref) * g_ref
        return g_update

In [20]:

def train_loop_gradient_selection_task0(model, dataloader, loss_fn, num_epoch, optimizer, device):

    model.train()
    model = model.to(device)

    #data we want to store in memory for the next task
    data_class0 = [] #(image, norme_gradient)
    data_class1 = [] #(image, norme_gradient)

    for epoch in range(num_epoch):

        epoch_loss = 0.0

        for images, labels, tasks in dataloader:
            images = images.to(device)
            labels = labels.to(device)
            tasks = tasks.to(device)

            optimizer.zero_grad()

            if (epoch < num_epoch - 1):
                outputs = model(images, tasks)

                loss  = loss_fn(outputs, labels)

                loss.backward()
                optimizer.step()
                epoch_loss += loss.item()



            else: # We will check the gradient norms only for the last epoch
                optimizer.zero_grad()
                for i in range(images.size(0)):
                    image_i = images[i].unsqueeze(0)
                    label_i = labels[i].unsqueeze(0)
                    task_i = tasks[i].unsqueeze(0)


                    for param in model.parameters():
                        param.requires_grad = False #we want to freeze all the layers exept the last in oreder to decrease the computational time
                    for param in model.heads.parameters():
                        param.requires_grad = True


                    model.heads.zero_grad()

                    pred_i = model(image_i, task_i)
                    loss_i = loss_fn(pred_i, label_i)

                    loss_i.backward()

                    #calculate the norm of the gradient
                    sum_squar = 0.0
                    for param in model.heads.parameters():
                        if param.grad is not None:
                            sum_squar += torch.sum(param.grad ** 2).item()
                    norm = sum_squar ** 0.5

                    img = images[i].cpu().detach()
                    lbl= labels[i].cpu().detach()

                    if lbl == 0:
                        data_class0.append((img,norm ))
                    elif lbl== 1:
                        data_class1.append((img,norm ))

                for param in model.parameters():
                    param.requires_grad = True

                #as the layers were frozen in the previous lines we do again the forward and backward for the last layer
                optimizer.zero_grad()
                outputs = model(images, tasks)
                loss = loss_fn(outputs, labels)
                loss.backward()
                optimizer.step()
                epoch_loss += loss.item()
        epoch_loss = epoch_loss/len(dataloader)


        print(f"Epoch {epoch+1}/{num_epoch}, Loss : {epoch_loss}")

    return data_class0, data_class1




In [ ]:
data_class_0, data_class_1 = train_loop_gradient_selection_task0(model = basic_model, dataloader = train_dataloader_AB_full , loss_fn = loss, num_epoch = num_epochs, optimizer = optimizer, device = device)

AttributeError: 'ResNet' object has no attribute 'heads'

In [21]:

def select_k_percent(data_class0, data_class1, k_percent=0.05):
    #Take 2 lists of couples (image, norm_gradient), sort the couples by keeping the images with the highest gradient
    nb_0 = int(len(data_class0) * k_percent)
    nb_1 = int(len(data_class1) * k_percent)
    data_k_percent = []

    data_class0.sort(key=lambda x: x[1], reverse=True)

    for couple in data_class0[:nb_0]:
        data_k_percent.append((couple[0], 0))


    data_class1.sort(key=lambda x: x[1], reverse=True)

    for couple in data_class1[:nb_1]:
        data_k_percent.append((couple[0], 1))


    return data_k_percent

In [22]:

def train_loop_gradient_selection_task1(model, dataloader, data_k_percent, loss_fn, num_epoch, optimizer, device, batch_size_mem=32):
    model.train()
    model = model.to(device)

    for epoch in range(num_epoch):
        epoch_loss = 0.0

        for images, labels, tasks in dataloader:
            images = images.to(device)
            labels = labels.to(device)
            tasks = tasks.to(device)

            optimizer.zero_grad()
            outputs = model(images, tasks)
            loss_task1 = loss_fn(outputs, labels)
            loss_task1.backward()

            g_1 = []
            for param in model.parameters():
                if param.grad is not None:
                    g_1.append(param.grad.view(-1))
            g_1 = torch.cat(g_1)


            batch_mem = random.sample(data_k_percent, min(batch_size_mem, len(data_k_percent)))

            mem_images = torch.stack([couple[0] for couple in batch_mem]).to(device)
            mem_labels = torch.tensor([couple[1] for couple in batch_mem]).to(device)
            # Pour la tâche 0, l'identifiant de tâche est 0 (important pour ton architecture multi-head)
            mem_tasks = torch.zeros_like(mem_labels).to(device)

            # On nettoie les gradients du modèle pour calculer proprement ceux du passé
            # (Ne t'inquiète pas, on a sauvé g_new juste au-dessus !)
            model.zero_grad()

            outputs_mem = model(mem_images, mem_tasks)
            loss_mem = loss_fn(outputs_mem, mem_labels).mean()
            loss_mem.backward()

            # On sauvegarde les gradients du passé (Tâche 0)
            g_ref = []
            for param in model.parameters():
                if param.grad is not None:
                    g_ref.append(param.grad.view(-1))
            g_ref = torch.cat(g_ref)

            # ---------------------------------------------------------
            # ÉTAPE 3 : LA PROJECTION A-GEM (La formule mathématique)
            # ---------------------------------------------------------
            # On calcule le produit scalaire entre le nouveau gradient et le passé
            g_update =project_gradient(g_1, g_ref)

            if not (g_update == g_1):
                # On réinjecte ce gradient projeté tout propre dans notre modèle
                index = 0
                for param in model.parameters():
                    if param.grad is not None:
                        num_el = param.grad.numel()
                        # On redonne au vecteur plat sa forme d'origine (matrice de poids)
                        param.grad.copy_(g_projected[index:index + num_el].view_as(param.grad))
                        index += num_el
            else:
                # S'il n'y a pas d'interférence, on remet les gradients d'origine de la Tâche 1
                index = 0
                for param in model.parameters():
                    if param.grad is not None:
                        num_el = param.grad.numel()
                        param.grad.copy_(g_new[index:index + num_el].view_as(param.grad))
                        index += num_el

            # ÉTAPE 4 : MISE À JOUR DES POIDS
            optimizer.step()
            epoch_loss += loss_task1.item()

        print(f"Epoch {epoch+1}/{num_epoch} (Tâche 1 avec A-GEM) - Loss : {epoch_loss/len(dataloader_task1):.4f}")

In [ ]:
### A partir d'ici c'set la partie 2 du cod . Avant ici c'est la partoe 1 du code . La partie 2 se foculise ecxcluisiovement sur les methodes d'entrainement eavec A-GEM . La partie 1 se focalise sur les prerequis pour la constructui des datstatset , dataloader et autrse et surtout sur le continual leanring sans A -GEM

In [72]:
from torch.utils.data import TensorDataset
from sklearn.cluster import KMeans
from torch.nn.utils import parameters_to_vector
from tqdm import tqdm

In [57]:
os.environ['CUDA_LAUNCH_BLOCKING'] = "1"
os.environ["TORCH_USE_CUDA_DSA"] = "1"

In [24]:


classe = {'buildings' : 0, 'forest' : 1, 'glacier' : 2, 'mountain' : 3, 'sea' : 4, 'street' : 5}
# To do the reverse path
label_to_classname= {0: 'buildings', 1: 'forest', 2: 'glacier', 3: 'mountain', 4: 'sea', 5: 'street'}

class IntelDataset(Dataset):
    def __init__(self, df, transform=None, base_dir=None, label_map=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform
        self.base_dir = base_dir # This will now be the task-specific img folder
        self.label_map = label_map # Conversion dictionary, ex: {3: 1, 1: 0}

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_name = row['image_name']
        label = int(row['label'])

        # No need to get class_name from label_to_classname if base_dir is already specific
        # The base_dir should be something like /content/my_intel_dataset/seg_train/seg_train/df_buildings_sea/img

        # Remapping so the label is 0 or 1 for the binary model
        if self.label_map is not None:
            label = self.label_map[label]

        # Simplified img_path construction
        img_path = os.path.join(self.base_dir, img_name)
        image = Image.open(img_path).convert('RGB')

        if self.transform:
            image = self.transform(image)

        return image, torch.tensor(label, dtype=torch.long)

In [71]:
#### GLOBAL PARAMETERS FOR PART 2 ####

NB_EPOCH=3
epochs_per_task=2
memory_size_per_task=300
batch_size=32

classe = {'buildings' : 0, 'forest' : 1, 'glacier' : 2, 'mountain' : 3, 'sea' : 4, 'street' : 5}
device="cuda" if torch.cuda.is_available() else "cpu"

# Mapping for each task to local head indices [0, 1]
task_mappings = {
    0: {0: 0, 4: 1},  # Task 0: Buildings -> 0, Sea -> 1
    1: {1: 0, 3: 1}   # Task 1: Forest -> 0, Mountain -> 1
}

tasks_definition = {
    0: [0, 4],
    1: [1, 3]
}

In [58]:
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

def create_mapped_loader(df, task_id, root_dir):

    # I use Laura's transform pipeline in my part (2e part)
    ds = IntelDataset(df, transform=transform, base_dir=root_dir, label_map=None)
    return DataLoader(ds, batch_size=32, shuffle=False)

#Dictionary containing the good dataloader for the test
dict_test_loader = {
    0: create_mapped_loader(label_AB_test, 0, path_img_AB_test),
    1: create_mapped_loader(label_CD_test, 1, path_img_CD_test),
}

In [60]:
import numpy as np
import torch
from torch.utils.data import TensorDataset, DataLoader
from sklearn.cluster import KMeans


class EpisodicMemory:
  def __init__(self, max_memories_sample=1000):
    """
    max_memory_sample : the max images to keep for each past task
    """
    self.max_memories_sample=max_memories_sample
    self.memory_dataset=None   # No TensorDataset yet

  def _extract_features(self,model,dataset,device):
    """
    extract the features of the layer just before the last layer
    """
    # switch to eval mode
    model.eval()

    # We take as feature extractor the whole model except the last layer
    backbone = torch.nn.Sequential(*list(model.children())[:-1]).to(device)
    features_list=[]
    temp_loader=DataLoader(dataset, batch_size=64,shuffle=False)

    with torch.no_grad():
      for images, _ in temp_loader:
        images=images.to(device)
        # the features go into the backbone to have the features
        features=backbone(images)
        features = features.view(features.size(0), -1)
        features_list.append(features.cpu().numpy())

    model.train()
    return np.concatenate(features_list, axis=0)

  def KMeans_sample_extractor(self,task_dataset, model=None,device='cuda'):
    """
    We extract the feature to choose the best image for each cluster using KMeans
    """
    budget = min(self.max_memories_sample, len(task_dataset))

    if model is not None:
      print("Extraction of the features with KMeans")
      features=self._extract_features(model,task_dataset,device)

      print(f'We use KMeans with K={budget}')

      # Use of KMEans to create clusters
      kmeans = KMeans(n_clusters=budget, n_init='auto', random_state=42)
      cluster_labels = kmeans.fit_predict(features)
      # the centroids
      centroids = kmeans.cluster_centers_

      # Selection of the best image for each cluster
      selected_indices=[]
      for i in range(budget):
        idx_in_cluster = np.where(cluster_labels == i)[0]
        if len(idx_in_cluster) > 0:
          cluster_features = features[idx_in_cluster]

          # We compute the distances
          distances = np.linalg.norm(cluster_features - centroids[i], axis=1)
          # The best indices
          best_local_idx = np.argmin(distances)
          selected_indices.append(idx_in_cluster[best_local_idx])


      # extraction of the tensors (images,labels) linked to these selected indices
      full_loader = DataLoader(task_dataset, batch_size=len(task_dataset), shuffle=False)
      all_images, all_labels = next(iter(full_loader))

      # Final selected images and labels
      images_from_samples = all_images[selected_indices]
      labels_sample = all_labels[selected_indices]

      print("K-Means ended")

      return images_from_samples,labels_sample


  def store_memories(self,task_dataset,id_task,method="random"):
    """
    extracts a random sample from the task that has just finished
    Method: “random” for the standard random method, “KMeans” for the K-means method
    """
    if method=="KMeans":
      images_from_samples,labels_sample=self.KMeans_sample_extractor(task_dataset,model,device)

    else:
      budget = min(self.max_memories_sample, len(task_dataset))

      # We create a temporary dataloader with batchsize=self.max_memories_sample
      temp_loader=DataLoader(task_dataset, batch_size=budget,shuffle=True)
      images_from_samples,labels_sample = next(iter(temp_loader))

    task_ids = torch.full((len(labels_sample),), id_task, dtype=torch.long)


    if self.memory_dataset is None:

      # Merge the images and the labels in a tensordataset
      self.memory_dataset=TensorDataset(images_from_samples,labels_sample, task_ids)

    else:
      old_labels=self.memory_dataset.tensors[1]
      old_images=self.memory_dataset.tensors[0]
      old_task_ids = self.memory_dataset.tensors[2]

      # We merge the past and the present

      new_images=torch.cat([old_images,images_from_samples] )
      new_labels=torch.cat([old_labels,labels_sample])
      new_task_ids = torch.cat([old_task_ids, task_ids])

      self.memory_dataset=TensorDataset(new_images,new_labels,new_task_ids)

      print(f"-> A-GEM Buffer updated. Total memory size: {len(self.memory_dataset)} images.")



  def get_reference_batch(self, batch_size):
    if self.memory_dataset is None:
      return None

    ref_loader=DataLoader(self.memory_dataset, batch_size=batch_size,shuffle=True)
    return next(iter(ref_loader))


In [61]:
def train_agrem_step(model, optimizer, criterion, current_batch, batch_ref, device, id_task):
    """
    Train the model using A-GEM for one step, handling Multi-Head gradients securely.
    """
    current_imgs, current_labels = current_batch
    current_imgs, current_labels = current_imgs.to(device), current_labels.to(device)


    img_ref, label_ref, task_id_ref = batch_ref
    img_ref, label_ref, task_id_ref = img_ref.to(device), label_ref.to(device), task_id_ref.to(device)

    # 1. Compute g (gradient for the current task)
    optimizer.zero_grad()
    current_output = model(current_imgs, id_task)
    current_loss = criterion(current_output, current_labels)
    current_loss.backward()

    # We replace the 'None by zeros' for more security
    grads = []
    for p in model.parameters():
        if p.requires_grad:
            if p.grad is not None:
                grads.append(p.grad.view(-1).clone())
            else:
                grads.append(torch.zeros_like(p).view(-1))
    g = torch.cat(grads)

    # 2. Compute g_ref (gradient for memory/reference batch)
    optimizer.zero_grad()

    #It prevent problems with the BathNorm (corruption of the memory)
    model.eval()

    # We calculate the loss
    unique_tasks = torch.unique(task_id_ref)
    loss_ref = 0
    for t in unique_tasks:
        mask = (task_id_ref == t)
        img_t = img_ref[mask]
        label_t = label_ref[mask]

        output_t = model(img_t, id_task=t.item())
        loss_t = criterion(output_t, label_t)
        # Pondération de la loss selon le nombre d'images de cette tâche
        loss_ref += loss_t * (img_t.size(0) / img_ref.size(0))

    loss_ref.backward()

    # Return in train mode
    model.train()

    grads_ref = []
    for p in model.parameters():
        if p.requires_grad:
            if p.grad is not None:
                grads_ref.append(p.grad.view(-1).clone())
            else:
                grads_ref.append(torch.zeros_like(p).view(-1))
    g_ref = torch.cat(grads_ref)


    # I use Laura's function to do A-GEM Projection
    g_update=project_gradient(g,g_ref)
    g=g_update

    # 4. Manually update gradients in the model
    optimizer.zero_grad()
    index = 0
    for p in model.parameters():
        if p.requires_grad:
            num_params = p.numel()
            p.grad = g[index:index + num_params].view(p.shape).clone()
            index += num_params

    optimizer.step()
    return current_loss.item()

In [62]:
def train_standard_step(model, optimizer, criterion, current_batch, device, id_task):
  """
  Train the model WITHOUT using A-GEM model for one step.
  """
  current_imgs, current_labels = current_batch
  current_imgs, current_labels = current_imgs.to(device), current_labels.to(device)

  optimizer.zero_grad()
  predictions = model(current_imgs, id_task)
  loss = criterion(predictions, current_labels)
  loss.backward()
  optimizer.step()

  return loss.item()

In [65]:
def evaluate_continual_eval(model, test_loaders, list_of_learned_task, device):
    """
    Function to evaluate the performances : accuracy and loss on the test set
    """
    model.eval()
    results = {}

    with torch.no_grad():
        for id_task in list_of_learned_task:
            specific_test_loader = test_loaders[id_task]
            total_correct = 0
            total_loss = 0.0
            total_samples = 0

            for images, labels in specific_test_loader:
                images, labels = images.to(device), labels.to(device)

                logits = model(images, id_task)
                loss = criterion(logits, labels)

                _, preds = torch.max(logits, dim=1)
                total_correct += (preds == labels).sum().item()
                total_loss += loss.item() * labels.size(0)
                total_samples += labels.size(0)
            #Dict with the performance
            results[id_task] = {
                'acc': total_correct / total_samples,
                'loss': total_loss / total_samples
            }

    model.train()
    return results

In [67]:
from tqdm import tqdm

def train_agem_global(model, optimizer, criterion, dict_test_loader, device='cuda'):
  """
  Final function to train the model with the A-GEM method (for all the step)
  """

  # Initialize memory buffer to store samples from previous tasks
  buffer_agem = EpisodicMemory(max_memories_sample=memory_size_per_task)
  list_of_learned_task = []
  model.to(device)

  for id_task, classes in tasks_definition.items():
    print(f"\n--- Task {id_task} (Classes: {classes}) ---")

    #We do Local mapping: map original classes to 0 and 1 for the specific task head
    local_map = {classes[0]: 0, classes[1]: 1}

    # We Select only images belonging to the current task classes
    #current_df = label_ABCD_train[label_ABCD_train['label'].isin(classes)]
    current_df = label_ABCD_train[label_ABCD_train['task_id'] == id_task]

    # Determine the correct image base directory for the current task
    if id_task == 0:
        current_task_img_path = path_img_AB_train
    elif id_task == 1:
        current_task_img_path = path_img_CD_train
    else:
        raise ValueError(f"Unknown task ID: {id_task}")


    current_dataset = IntelDataset(
        current_df,
        transform=transform,
        label_map=None,
        base_dir=current_task_img_path # Use the task-specific path here
    )
    current_loader = DataLoader(current_dataset, batch_size=batch_size, shuffle=True)

    for epoch in range(1, epochs_per_task + 1):
      model.train()
      for imgs, labels in tqdm(current_loader, desc=f"Epoch {epoch}"):
        # Get a reference batch from the memory buffer
        batch_ref = buffer_agem.get_reference_batch(batch_size=batch_size)

        if batch_ref is not None:

          imgs_ref, labels_ref, task_ids_ref = batch_ref

          # Use A-GEM step if we have samples from previous tasks

          train_agrem_step(model, optimizer, criterion, (imgs, labels), (imgs_ref, labels_ref,task_ids_ref), device, id_task)
        else:
          # Standard training for the first task
          train_standard_step(model, optimizer, criterion, (imgs, labels), device, id_task)

    # Save some images to the buffer after finishing the task
    buffer_agem.store_memories(current_dataset,id_task=id_task ,method='random')
    list_of_learned_task.append(id_task)

    # Evaluate the model on all tasks learned so far
    results = evaluate_continual_eval(model, dict_test_loader, list_of_learned_task, device)
    print(f"\n Result after task {id_task} :")
    for t_id, metrics in results.items():
      print(f"   * Task {t_id} -> Accuracy: {metrics['acc']*100:.2f}% | Loss: {metrics['loss']:.4f}")

In [68]:
# Clear GPU cache
torch.cuda.empty_cache()

class MultiHeadResNet(nn.Module):
    def __init__(self, original_resnet):
        super(MultiHeadResNet, self).__init__()
        # Backbone: all layers except the final FC
        self.features = nn.Sequential(*list(original_resnet.children())[:-1])
        # One head (Linear layer) per task, each outputting 2 classes
        self.heads = nn.ModuleList([nn.Linear(512, 2), nn.Linear(512, 2)])

    def forward(self, x, id_task):
        # Changed parameter name task_id to id_task to fix the TypeError
        x = self.features(x)
        x = torch.flatten(x, 1)
        x = self.heads[id_task](x)
        return x

# Initialize model
device = "cuda" if torch.cuda.is_available() else "cpu"
base_model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
model = MultiHeadResNet(base_model).to(device)

criterion = nn.CrossEntropyLoss()
# Optimizer tracks all parameters; A-GEM step handles the gradient logic
optimizer = optim.SGD(model.parameters(), lr=0.001, momentum=0.9)

# Start training
train_agem_global(model, optimizer, criterion, dict_test_loader, device)


--- Task 0 (Classes: [0, 4]) ---


Epoch 2: 100%|██████████| 140/140 [00:13<00:00, 10.03it/s]



 Result after task 0 :
   * Task 0 -> Accuracy: 99.26% | Loss: 0.0202

--- Task 1 (Classes: [1, 3]) ---


Epoch 2: 100%|██████████| 150/150 [00:26<00:00,  5.58it/s]


-> A-GEM Buffer updated. Total memory size: 600 images.

 Result after task 1 :
   * Task 0 -> Accuracy: 99.05% | Loss: 0.0256
   * Task 1 -> Accuracy: 99.40% | Loss: 0.0197
